In [1]:
import pandas as pd
import re
import jieba
import nltk
from nltk.tokenize import TreebankWordTokenizer
from nltk.corpus import stopwords as nltk_stopwords
from stopwords import get_stopwords

# 1- Setup 



In [2]:
#nltk.data.path.append('/Users/rachelliu/nltk_data')
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/abeltewodros/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/abeltewodros/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# 2- Load CSV files 

In [3]:
# Load CSV files
df_en = pd.read_csv('data/Data_English.csv')
df_ar = pd.read_csv('data/Data_Arabic.csv')
df_zh = pd.read_csv('data/Data_Mandarin.csv')
df_fa = pd.read_csv('data/Data_Farsi.csv')
df_fr = pd.read_csv('data/Data_French.csv')
df_hi = pd.read_csv('data/Data_Hindi.csv')
df_id = pd.read_csv('data/Data_Indonesian.csv')
df_pt = pd.read_csv('data/Data_Portugese.csv')
df_ru = pd.read_csv('data/Data_Russian.csv')
df_es = pd.read_csv('data/Data_Spanish.csv')
df_tr = pd.read_csv('data/Data_Turkish.csv')
df_uk = pd.read_csv('data/Data_Ukranian.csv')
df_ur = pd.read_csv('data/Data_Urdu.csv')

# 3- Combine ALl Data

In [4]:
df = pd.concat([df_en, df_ar, df_zh, df_fa, df_fr, df_hi, df_id, df_pt, df_ru, df_es, df_tr, df_uk, df_ur], ignore_index=True)

# 4- NLTK Stopwords Map 

In [5]:
nltk_lang_map = {
    'en': 'english', 'ar': 'arabic', 'es': 'spanish', 'pt': 'portuguese',
    'fr': 'french', 'ru': 'russian', 'tr': 'turkish', 'id': 'indonesian'
}

nltk_stopwords_dict = {
    lang_code: set(nltk_stopwords.words(nltk_lang_map[lang_code]))
    for lang_code in nltk_lang_map
}


# 5- Extended Stopwords 

In [6]:
# Arabic (ar)
extended_arabic_stopwords = set(nltk_stopwords.words('arabic')).union({
    'التي', 'الذي', 'الذين', 'اللذان', 'اللذين', 'اللائي', 'اللاتي',
    'كأن', 'لعل', 'حتى', 'إذن', 'قد', 'لن', 'لم', 'ما', 'من', 'في',
    'عن', 'على', 'إلى', 'أن', 'إن', 'لكن', 'أو', 'بل', 'ثم', 'ف', 'و', 'أم'
})

# Mandarin (zh)
extended_mandarin_stopwords = set("""
的 了 和 是 就 都 而 及 與 著 或 一個 沒有 我們 你們 他們 她們 是否 所以 而且 並且 如果 但是 因為 這樣 這個 那個 以及 啊 嗎 吧 呢 嘛 哦 哇
也 很 被 要 在 有 就是 就算 誰 哪 那 什麼 怎麼 怎樣 因為 此 此外 仍然 每個 對於 通過
""".split())

# Farsi (fa)
extended_farsi_stopwords = set("""
اما اگر از اش با باشد باشدند بودن بود بودم بودند باشد بکن بکنید بکند برای برسد بس باشد بتوان بدون بعضی بعد به بی تا تمام تنها توج توجه تیز ترین ترین ها هاست هایی همچنین حتی خاص خود خیلی خیلیی در دریافت دارد داشت داشتم داشتند داشته در دیگر را راستی راحت راحتی راه روز روزی روی شد شدند شده شدن شدم شدید شدیدا شوید شاید شود شون شناسی صورت طبق طول طی ظهر ظهرها عجب عجیب علی رغم علیه عمل فقط فرا فرق فوق قبل قبلا قد قدری قرار قطعا لحظه لذا لذاست لذاً لک لیک لیل مان مانم مانید مانیم مانند مارس مثلا مثال مثالاً مثل مثلاً مدتی مردم مرسی مشکل مشکلات مطمئنا مطرح معین مشخص معلومات معکوس معلوم مفید مقداری ممکن می میخواهم میتوانم میکند نا ناگه نداشت ندارد نماید نمی نمیخواهید نمیخواهیم نمیخواهم نمیخواهند نمی‌کنید نیست نیستند نیک هنگامی واکنش واکنشی وجود واقعی واقعیاً واقعیت واقعاً ولی وی نیز یک
""".split())

# Urdu (ur)
extended_urdu_stopwords = set("""
اور کے کو میں یہ کہ ہے ہوں تھا تھے ہیں نہیں پر سے بھی نے تو تھا تک کی کریں کرو کر رہا رہی رہے جیسے جب جس جن جنہیں جنہوں جو چکا چکی چکے کیا کون کسی کس کیسے کیوں پھر کیونکہ شاید تاکہ تاکہ والا والے والی والوں تک تم ہم تمہارے ہمارا ہماری ہمارے میری میرا میرے آپ آپکا آپکی آپکے وہ ان انہیں انہوں انکے انکی انکا اسے اسے اسے اسکا اسکی اسکے اسی اسی اس میں تھا تھے تھیں تھوڑی تھوڑا تھوڑے زیادہ سب کوئی کئی کچھ یہ وہ وہی یہیں یہاں وہاں کیسے کیسا کسی کو نہ ہر وغیرہ علاوہ بعد پہلے علاوہ مز مزے مزہ لگا لگے لگا لگا ہو ہوئ ہوئ ہوئیں ہوئے ہوا ہوجائے ہوگئے ہوگئی ہوجاتی ہوچکا ہوچکی ہوچکے ہو رہا ہو رہی ہو رہے
""".split())

# Treebank Tokenizer for English
--- Tokenizer specifically designed fro English syntax (punctuation, contractions)

In [7]:
en_tokenizer = TreebankWordTokenizer()

# Cleaners

In [8]:
def clean_english(text):
    if pd.isna(text): return text #check if text is missing 
    words = en_tokenizer.tokenize(text.lower()) #convert to lowercase, use treebank tokenizer to split text into tokens
    words = [word for word in words if word.isalpha()] #keep only alphabetic words
    return ' '.join([w for w in words if w not in nltk_stopwords_dict['en']]) #remove stopwords and return clean text as string 

def clean_mandarin(text):
    if pd.isna(text): return text
    words = jieba.lcut(str(text)) # Uses jieba to segment Chinese text into words. Chinese has no spaces
    return ' '.join([word for word in words if word not in extended_mandarin_stopwords and word.strip()])

#Handles all other languages 
def generic_cleaner(text, lang):
    if pd.isna(text): return text
    words = re.findall(r'\b\w+\b', text.lower())
    
    if lang == 'ar': #Arabic, Mandarin, Farsi, and Urdu get extended manual lists
        stopword_set = extended_arabic_stopwords
    elif lang == 'zh':
        stopword_set = extended_mandarin_stopwords
    elif lang == 'fa':
        stopword_set = extended_farsi_stopwords
    elif lang == 'ur':
        stopword_set = extended_urdu_stopwords
    elif lang in nltk_stopwords_dict: #NLTK stopwords for known languages: Spanish, Porugese, French, Russian, turkish, indonesian
        stopword_set = nltk_stopwords_dict[lang]

    #Any fallback language uses the stopwords package ( Ukranian, Hindi)
    else:
        try:
            stopword_set = set(get_stopwords(lang))
        except:
            stopword_set = set()

    return ' '.join([w for w in words if w not in stopword_set])

def multilingual_cleaner(row):
    lang = row['language_code']
    text = row['messages']
    if lang == 'en':
        return clean_english(text)
    elif lang == 'zh':
        return clean_mandarin(text)
    else:
        return generic_cleaner(text, lang)

# --- Apply to DataFrame ---

In [9]:
df['messages_cleaned'] = df.apply(multilingual_cleaner, axis=1)

Building prefix dict from the default dictionary ...
Dumping model to file cache /var/folders/3m/9hshj2x97_vdrql6qv6dngwc0000gn/T/jieba.cache
Loading model cost 0.294 seconds.
Prefix dict has been built successfully.


In [10]:
languages_to_preview = ['en', 'ar', 'zh', 'fa', 'ru', 'es', 'tr', 'uk', 'ur', 'fr', 'hi', 'pt', 'id']

for lang in languages_to_preview:
    print(f"🔤 Language: {lang.upper()}")
    subset = df[df['language_code'] == lang][['messages', 'messages_cleaned']].head(3)
    print(subset.to_string(index=False))  # prettier printing without row numbers
    print("-" * 60)


🔤 Language: EN
                                                                                                             messages                                      messages_cleaned
I have been using alcohol to numb my pain, but I know it’s not the answer. How do I find healing through God instead? using alcohol numb pain know find healing god instead
                              I have been through detox before, but I always go back. How do I make a lasting change?                   detox always go make lasting change
                                     I feel like my past defines me. How do I embrace the future that God has for me?             feel like past defines embrace future god
------------------------------------------------------------
🔤 Language: AR
                                                                                                       messages                                                                             messages_cleaned
لقد كنت أعاني من

# Export 


In [11]:

df.to_csv("GMO_Cleaned_Messages_Final.csv", index=False)

#-- Checking for Errors

In [12]:
print(df['language_code'].value_counts())


language_code
en    400
ar    100
zh    100
fa    100
hi    100
id    100
ru    100
es    100
tr    100
uk    100
ur    100
fr     99
pt     99
Name: count, dtype: int64


In [13]:
print(df_ru.columns)

Index(['contact_uuid', 'first_name', 'last_name', 'source', 'language_code',
       'decision', 'country', 'region', 'state', 'messages', 'topic'],
      dtype='object')


In [14]:
from collections import Counter
from tensorflow.keras.preprocessing.text import Tokenizer

In [15]:
def counter(text):
    count=Counter()
    for i in text.messages_cleaned:
        for word in i.split():
            count[word]+=1
    return count

In [16]:
testing=counter(df.head(400))
print(testing)
num_word=len(testing)
print(num_word)

Counter({'feel': 121, 'like': 64, 'want': 38, 'god': 34, 'know': 32, 'life': 29, 'faith': 23, 'lost': 22, 'everything': 22, 'help': 20, 'addiction': 19, 'find': 17, 'keep': 17, 'need': 16, 'peace': 13, 'family': 13, 'every': 13, 'spiritual': 13, 'scared': 13, 'time': 11, 'home': 11, 'spouse': 11, 'anymore': 11, 'one': 10, 'pain': 9, 'past': 9, 'afraid': 9, 'falling': 9, 'partner': 9, 'love': 9, 'still': 8, 'trying': 8, 'alone': 8, 'start': 8, 'back': 8, 'apart': 8, 'marriage': 8, 'spiritually': 8, 'stop': 8, 'nothing': 8, 'day': 8, 'tired': 8, 'future': 7, 'believe': 7, 'something': 7, 'fear': 7, 'constantly': 7, 'thoughts': 7, 'pray': 7, 'someone': 7, 'everyone': 7, 'think': 7, 'drugs': 7, 'much': 7, 'go': 6, 'people': 6, 'stay': 6, 'free': 6, 'without': 6, 'strong': 6, 'trust': 6, 'even': 6, 'hard': 6, 'kids': 6, 'feels': 6, 'way': 6, 'could': 6, 'dark': 6, 'talk': 6, 'fighting': 6, 'get': 6, 'make': 5, 'used': 5, 'rebuild': 5, 'really': 5, 'support': 5, 'let': 5, 'control': 5, 'chil

In [17]:
token=Tokenizer(num_words=num_word)
token.fit_on_texts(testing)
word_i=token.word_index
print(word_i)

{'using': 1, 'alcohol': 2, 'numb': 3, 'pain': 4, 'know': 5, 'find': 6, 'healing': 7, 'god': 8, 'instead': 9, 'detox': 10, 'always': 11, 'go': 12, 'make': 13, 'lasting': 14, 'change': 15, 'feel': 16, 'like': 17, 'past': 18, 'defines': 19, 'embrace': 20, 'future': 21, 'war': 22, 'peace': 23, 'promises': 24, 'used': 25, 'believe': 26, 'addiction': 27, 'made': 28, 'question': 29, 'faith': 30, 'afraid': 31, 'disappointing': 32, 'people': 33, 'helping': 34, 'stay': 35, 'accountable': 36, 'lost': 37, 'everything': 38, 'rebuild': 39, 'life': 40, 'help': 41, 'want': 42, 'free': 43, 'deserve': 44, 'still': 45, 'save': 46, 'weak': 47, 'recovery': 48, 'rely': 49, 'strength': 50, 'trying': 51, 'quit': 52, 'secret': 53, 'ask': 54, 'without': 55, 'shame': 56, 'one': 57, 'understands': 58, 'going': 59, 'really': 60, 'see': 61, 'struggles': 62, 'sober': 63, 'strong': 64, 'forgives': 65, 'truly': 66, 'accept': 67, 'grace': 68, 'serve': 69, 'disqualifies': 70, 'use': 71, 'something': 72, 'good': 73, 'bat

In [18]:
X=token.texts_to_sequences(df.messages_cleaned)
print(X)

[[1, 2, 3, 4, 5, 6, 7, 8, 9], [10, 11, 12, 13, 14, 15], [16, 17, 18, 19, 20, 21, 8], [16, 17, 22, 6, 23, 24], [25, 26, 8, 27, 28, 29, 6, 30], [31, 32, 33, 34, 35, 36], [37, 38, 39, 40, 41], [42, 43, 27, 16, 17, 44, 8, 45, 46], [16, 47, 48, 49, 50, 9], [51, 52, 53, 31, 54, 41, 55, 56], [16, 17, 57, 58, 59, 8, 60, 61, 62], [63, 45, 35, 64, 30], [5, 8, 65, 45, 16, 66, 67, 68], [42, 69, 8, 16, 17, 18, 70, 45, 71, 72, 73], [16, 17, 74, 27, 75, 76, 8, 60], [77, 78, 24, 79, 39, 76], [37, 80, 81, 82, 16, 83, 26, 45, 21], [16, 17, 27, 28, 84, 26, 8, 45, 85], [86, 87, 88, 45, 89, 30, 90, 48], [16, 17, 37, 91, 8, 92, 93, 40], [37, 94, 4, 76, 95, 96, 97], [16, 98, 8, 3, 99], [31, 100, 101, 92, 30, 41, 102, 103], [42, 26, 104, 105, 57, 106, 30], [107, 33, 108, 16, 17, 37, 39, 109, 30], [16, 17, 110, 111, 94, 37, 112, 113, 55, 114, 115], [116, 117, 79, 5, 118, 30, 41, 119, 120, 21], [16, 121, 122, 6, 123, 8], [124, 125, 126, 112, 30, 64, 117, 127], [16, 17, 18, 19, 101, 43, 8, 60, 128, 129, 130], [1

In [28]:
label_to_index={'General':0,'Family and Marriage':1,'Crisis':2,'Spiritual Warfare':3,'Alcoholism and Drug Addiction':4,'Suicide':5}
df_en['labeled_topic']=df.topic.map(label_to_index)
print(df_en['labeled_topic'])
Y=df_en['labeled_topic'].values
print(Y)
num_classes=len(set(Y))
print(num_classes)



0      4.0
1      4.0
2      4.0
3      4.0
4      4.0
      ... 
395    3.0
396    5.0
397    2.0
398    5.0
399    0.0
Name: labeled_topic, Length: 400, dtype: float64
[4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3.
 3. 3. 3. 3. 2. 2. 4. 2. 2. 0. 3. 3. 4. 0. 0. 4. 1. 0. 0. 1. 5. 0. 0. 0.
 4. 3. 1. 1. 2. 1. 0. 3. 4. 0. 2. 2. 2. 4. 4. 4. 2. 2. 3. 1. 0. 1. 4. 5.
 1. 1. 3. 5. 5. 0. 3. 4. 5. 0. 1. 2. 1. 1. 3. 2. 1. 2. 4. 1. 4. 3. 0. 0.
 4. 5. 2. 0. 3. 5. 1. 4. 2. 5. 5. 5. 5. 2. 0. 5. 5. 5. 0. 5. 2. 2. 5. 3.
 3. 5. 5. 4. 1. 4. 2. 5. 5. 1. 5. 1. 3. 4. 1. 4. 0. 5. 2. 1. 4. 2. 0. 1.
 4. 0. 5. 2. 5. 3. 5. 0. 4. 4. 5. 2. 0. 0. 0. 3. 4. 3. 2. 2. 4. 4. 1. 3.
 1. 4. 3. 1. 4. 3. 1. 0. 0. 1. 3. 0. 1. 0. 3. 3. 1. 2. 3. 5. 3. 2. 5. 3.
 1. 0. 1. 0. 3. 2. 3. 0. 0.

In [29]:
from tensorflow.keras.utils import to_categorical

Y_cat = to_categorical(Y, num_classes=num_classes)

In [20]:
import random

def fill_vectors_to_75(list_of_vectors):
    # Loop through each vector in the list
    for i in range(len(list_of_vectors)):
        vec = list_of_vectors[i]
        # Calculate how many zeros need to be added
        zeros_to_add = 75 - len(vec)
        
        # If the vector is already 75 or more elements, no change
        if zeros_to_add > 0:
            # Extend the vector by adding zeros
            vec.extend([0] * zeros_to_add)
    
    return list_of_vectors
X=fill_vectors_to_75(X[:100])
print(len(X))


100


In [33]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Embedding, Dense
import numpy as np

model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=100, input_length=75))
model.add(LSTM(128, return_sequences=False))
model.add(Dense(6, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [35]:
model.fit(np.array(X),Y_cat,epochs=50,batch_size=100,validation_split=0.2)
model.summary()

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.2750 - loss: 1.3778 - val_accuracy: 0.0000e+00 - val_loss: 4.9436
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.3000 - loss: 1.3766 - val_accuracy: 0.0000e+00 - val_loss: 5.1362
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.3000 - loss: 1.3808 - val_accuracy: 0.0000e+00 - val_loss: 5.3531
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.3000 - loss: 1.3760 - val_accuracy: 0.0000e+00 - val_loss: 5.5714
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - accuracy: 0.3000 - loss: 1.3705 - val_accuracy: 0.0000e+00 - val_loss: 5.7700
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.3000 - loss: 1.3718 - val_accuracy: 0.0000e+00 - val_loss: 5.9174
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.2750 - loss: 1.3752 - val_accuracy: 0.0000e+00 - val_loss: 6.0007
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.3000 - loss: 1.3741 - val_

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 75, 100)        │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 128)            │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,854,068 (7.07 MB)

 Trainable params: 618,022 (2.36 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,236,046 (4.72 MB)